# 📊 Eikon Data API Notebook

### 1. 使用Eikon

In [1]:
#!pip install eikon

In [2]:
# 导入库
import eikon as ek

# 设置 API Key（需要你在 Eikon / Workspace 里生成）
config = {
    'eikon': {
        'app_key': 'YOUR_APP_KEY_HERE'
    }
}

ek.set_app_key(config['eikon']['app_key'])

In [3]:
# 查看包信息
help(ek)

Help on package eikon:

NAME
    eikon - # coding: utf-8

PACKAGE CONTENTS
    Profile
    data_grid
    eikonError
    json_requests
    news_request
    streaming_session (package)
    symbology
    time_series
    tools
    vendor (package)

SUBMODULES
    cache
    desktop_session
    istream_callback
    itemstream
    session
    stream
    stream_connection
    streamingprice
    streamingprice_callback
    streamingprices

VERSION
    1.1.18

FILE
    d:\programfiles\anaconda\envs\quant\lib\site-packages\eikon\__init__.py




In [4]:
# 定义股票代码（Reuters Instrument Code）
symbols = ['AAPL.O', 'MSFT.O', 'GOOG.O']

# 获取时间序列数据（日线）
data = ek.get_timeseries(
    symbols,
    start_date='2020-01-01',
    end_date='2020-05-01',
    interval='daily',
    fields=['*']
)

2026-04-13 21:39:10,001 P[10752] [MainThread 9600] Error code 502 | Server Error: 
2026-04-13 21:39:10,001 P[10752] [MainThread 9600] HTTP request failed: EikonError-Server Error: 


EikonError: Error code 502 | Server Error: 

In [ ]:
# 查看数据结构（MultiIndex）
data.keys()

In [ ]:
# 单个股票数据类型
type(data['AAPL.O'])

In [ ]:
# 数据概览
data['AAPL.O'].info()

In [ ]:
# 查看最后几行
data['AAPL.O'].tail()

#### 分钟级数据（更高频）

In [ ]:
%%time

data = ek.get_timeseries(
    symbols,
    start_date='2020-08-14',
    end_date='2020-08-15',
    interval='minute',
    fields='*'
)

In [ ]:
# 查看 GOOG 某时间段数据
print(
    data['GOOG.O'].loc[
        '2020-08-14 16:00:00':'2020-08-14 16:04:00'
    ]
)

In [ ]:
# 每个股票打印部分数据
for sym in symbols:
    print('\n' + sym + '\n', data[sym].iloc[-300:-295])

#### Tick 级数据（超高频）

In [ ]:
%%time

data = ek.get_timeseries(
    symbols[0],
    start_date='2020-08-14 15:00:00',
    end_date='2020-08-14 15:30:00',
    interval='tick',
    fields=['*']
)

In [ ]:
# 数据结构
data.info()

In [ ]:
# 前几行
data.head()

In [ ]:
# 重采样为 30 秒
resampled = data.resample('30s', label='right').agg({
    'VALUE': 'last',   # 取最后价格
    'VOLUME': 'sum'    # 成交量求和
})

In [ ]:
# 查看结果
resampled.tail()

使用eikon访问structured data， ek.get_timeseries 是核心接口

支持多标的（symbols list）

支持多频率：
- tick
- minute
- hour
- daily
- weekly / monthly / yearly

返回 MultiIndex DataFrame

高频数据（tick）需要 resample 才能用于策略

In [5]:
# 大陆暂时无法顺利使用eikon，为了学习，这里模拟一个eikon来实现基本功能

### 2. 使用 Mock Eikon

In [6]:
# =========================================
# 1. 导入库
# =========================================
import pandas as pd
import numpy as np
from datetime import datetime, timedelta


# =========================================
# 2. 模拟 Eikon API
# =========================================
class MockEikon:
    
    def __init__(self):
        self.app_key = None
    
    def set_app_key(self, key):
        self.app_key = key
        print(f"[MockEikon] App key set: {key}")
    
    def get_timeseries(self, symbols, start_date, end_date, interval='daily', fields=['*']):
        
        if isinstance(symbols, str):
            symbols = [symbols]
        
        start = pd.to_datetime(start_date)
        end = pd.to_datetime(end_date)
        
        data = {}
        
        for sym in symbols:
            df = self._generate_data(sym, start, end, interval)
            data[sym] = df
        
        # 模拟 MultiIndex（和书里一致）
        result = pd.concat(data, axis=1)
        return result
    
    
    def _generate_data(self, symbol, start, end, interval):
        
        if interval == 'daily':
            dates = pd.date_range(start, end, freq='B')
        elif interval == 'minute':
            dates = pd.date_range(start, end, freq='1min')
        elif interval == 'tick':
            # 模拟不规则 tick
            n = 5000
            dates = [start + timedelta(seconds=np.random.rand()*1800) for _ in range(n)]
            dates = sorted(dates)
        else:
            raise ValueError("Unsupported interval")
        
        base = 100 + np.random.randn() * 10
        
        df = pd.DataFrame(index=dates)
        df['OPEN'] = base + np.random.randn(len(dates))
        df['HIGH'] = df['OPEN'] + abs(np.random.randn(len(dates)))
        df['LOW'] = df['OPEN'] - abs(np.random.randn(len(dates)))
        df['CLOSE'] = df['OPEN'] + np.random.randn(len(dates))
        df['VOLUME'] = np.random.randint(100, 10000, len(dates))
        df['COUNT'] = np.random.randint(1, 500, len(dates))
        
        return df.sort_index()


# =========================================
# 3. 初始化 API（对应书里 ek.set_app_key）
# =========================================
ek = MockEikon()

config = {
    'eikon': {
        'app_key': 'DUMMY_KEY_123'
    }
}

ek.set_app_key(config['eikon']['app_key'])

[MockEikon] App key set: DUMMY_KEY_123


#### 获取历史数据

In [7]:
# =========================================
# 4. 获取历史日线数据（对应书 In[39]~[44]）
# =========================================
symbols = ['AAPL.O', 'MSFT.O', 'GOOG.O']

data = ek.get_timeseries(
    symbols,
    start_date='2020-01-01',
    end_date='2020-05-01',
    interval='daily',
    fields=['*']
)

print(data.keys())
print(type(data['AAPL.O']))
print(data['AAPL.O'].info())
print(data['AAPL.O'].tail())

MultiIndex([('AAPL.O',   'OPEN'),
            ('AAPL.O',   'HIGH'),
            ('AAPL.O',    'LOW'),
            ('AAPL.O',  'CLOSE'),
            ('AAPL.O', 'VOLUME'),
            ('AAPL.O',  'COUNT'),
            ('MSFT.O',   'OPEN'),
            ('MSFT.O',   'HIGH'),
            ('MSFT.O',    'LOW'),
            ('MSFT.O',  'CLOSE'),
            ('MSFT.O', 'VOLUME'),
            ('MSFT.O',  'COUNT'),
            ('GOOG.O',   'OPEN'),
            ('GOOG.O',   'HIGH'),
            ('GOOG.O',    'LOW'),
            ('GOOG.O',  'CLOSE'),
            ('GOOG.O', 'VOLUME'),
            ('GOOG.O',  'COUNT')],
           )
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 88 entries, 2020-01-01 to 2020-05-01
Freq: B
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   OPEN    88 non-null     float64
 1   HIGH    88 non-null     float64
 2   LOW     88 non-null     float64
 3   CLOSE   88 non-n

#### 分钟级数据

In [8]:
# =========================================
# 5. （对应书 In[45]~[47]）
# =========================================
data_min = ek.get_timeseries(
    symbols,
    start_date='2020-08-14',
    end_date='2020-08-15',
    interval='minute',
    fields='*'
)

print(data_min['GOOG.O'].head())

for sym in symbols:
    print("\n", sym)
    print(data_min[sym].iloc[-5:])

                           OPEN        HIGH        LOW       CLOSE  VOLUME  \
2020-08-14 00:00:00   98.517467   98.753668  98.074320   99.910951    9686   
2020-08-14 00:01:00   99.606383   99.737076  99.537558  100.633798    6743   
2020-08-14 00:02:00   99.244489   99.352124  98.754276   98.404925    4672   
2020-08-14 00:03:00   98.458619   98.934696  97.710987   98.242508    2337   
2020-08-14 00:04:00  100.021670  100.442102  99.299037  100.686455    1293   

                     COUNT  
2020-08-14 00:00:00    121  
2020-08-14 00:01:00     59  
2020-08-14 00:02:00    282  
2020-08-14 00:03:00     54  
2020-08-14 00:04:00    175  

 AAPL.O
                           OPEN        HIGH         LOW       CLOSE  VOLUME  \
2020-08-14 23:56:00  105.523911  106.374688  104.223933  104.951377    5387   
2020-08-14 23:57:00  105.269862  105.632710  104.989656  106.698887    1278   
2020-08-14 23:58:00  106.261972  107.967241  105.529612  105.331093    1807   
2020-08-14 23:59:00  105.644347 

#### Tick 数据 + 高频数据结构

In [9]:
# =========================================
# 6. Tick 数据 + 高频数据结构（对应书 In[48]~[50]）
# =========================================
tick_data = ek.get_timeseries(
    'AAPL.O',
    start_date='2020-08-14 15:00:00',
    end_date='2020-08-14 15:30:00',
    interval='tick',
    fields=['*']
)

df_tick = tick_data['AAPL.O']

print(df_tick.info())
print(df_tick.head())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 5000 entries, 2020-08-14 15:00:00.541253 to 2020-08-14 15:29:59.748046
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   OPEN    5000 non-null   float64
 1   HIGH    5000 non-null   float64
 2   LOW     5000 non-null   float64
 3   CLOSE   5000 non-null   float64
 4   VOLUME  5000 non-null   int32  
 5   COUNT   5000 non-null   int32  
dtypes: float64(4), int32(2)
memory usage: 234.4 KB
None
                                 OPEN       HIGH        LOW      CLOSE  \
2020-08-14 15:00:00.541253  92.376312  93.935011  92.162052  92.660145   
2020-08-14 15:00:01.346952  92.514419  93.043627  91.679838  92.110013   
2020-08-14 15:00:01.619527  95.126496  96.256736  94.811392  96.497498   
2020-08-14 15:00:01.657924  91.196496  91.207280  88.807945  89.468084   
2020-08-14 15:00:02.078417  91.517348  91.910374  90.940782  93.537772   

                            VOLUME  COUNT  
20

#### Resample

In [10]:
# =========================================
# 7. Resample（核心：pandas 重采样）
# =========================================
resampled = df_tick.resample('30s').agg({
    'CLOSE': 'last',
    'VOLUME': 'sum'
})

print(resampled.tail())

                         CLOSE  VOLUME
2020-08-14 15:27:30  93.655338  362158
2020-08-14 15:28:00  91.914015  380813
2020-08-14 15:28:30  93.319097  399515
2020-08-14 15:29:00  96.541569  554594
2020-08-14 15:29:30  93.954936  526050
